In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (2), wrong dep version loaded (1), incompatible header (6))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (4), wrong dep version loaded (2), incompatible header (12))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
openGeometry("box.geo")

In [3]:
#openPreProcessor()

In [4]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [ ]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=true)

0

In [28]:
@time C = contact(u, master="master", slave="slave", leaf_size=1)

  0.157790 seconds (67.28 k allocations: 7.987 MiB)


Contact("slave" -> "master", 1149 candidate nodes, 603 active, stick=603, slip=0, G=(3447, 26790), C=(3447, 3447))

In [30]:
@time updateContact!(C, 1.3u)

  0.151049 seconds (67.27 k allocations: 8.183 MiB, 6.03% gc time)


Contact("slave" -> "master", 1149 candidate nodes, 712 active, stick=712, slip=0, G=(3447, 26790), C=(3447, 3447))

In [31]:
@time updateContact!(C, 1.01u)

  0.140559 seconds (67.27 k allocations: 8.180 MiB)


Contact("slave" -> "master", 1149 candidate nodes, 604 active, stick=604, slip=0, G=(3447, 26790), C=(3447, 3447))

In [34]:
for ls in (1, 2, 4, 8)
    @time contact(
        u,
        master="master",
        slave="slave",
        leaf_size=ls
    )
end

  0.179916 seconds (67.27 k allocations: 7.986 MiB, 2.27% gc time)
  0.180727 seconds (64.32 k allocations: 7.884 MiB)
  0.228830 seconds (62.72 k allocations: 7.823 MiB)
  0.281171 seconds (61.92 k allocations: 7.789 MiB)


In [9]:
C.gap

elementwise ScalarField
[[0.11748664025768041; 0.10125044128166245; … ; 0.0969625160407129; 0.10520352967442786;;], [0.10129836059057813; 0.11748664025768041; … ; 0.10520352967442786; 0.09698953572322645;;], [0.10129954162619699; 0.1175228377827056; … ; 0.10521551140617136; 0.09698453523752289;;], [0.1175228377827056; 0.10124444273701104; … ; 0.09695981074649018; 0.10521551140617136;;], [0.10124496497632966; 0.11748531846133077; … ; 0.10520526864880729; 0.0969591308321874;;], [0.11748531846133077; 0.10130639094925331; … ; 0.0969945785018007; 0.10520526864880729;;], [0.11742801117433892; 0.10125972502619231; … ; 0.09695208537883136; 0.10517276738958885;;], [0.10128270706190516; 0.11742801117433892; … ; 0.10517276738958885; 0.09697797918354467;;], [0.08796594129353001; 0.10129954162619699; … ; 0.09009218477198189; 0.0826537259375859;;], [0.07806130360761077; 0.10129954162619699; … ; 0.09698453523752289; 0.0854427817904895;;]  …  [0.07807802546680531; 0.07806759625718933; … ; 0.0855009632

In [10]:
C.G[:,:]

3447×26790 SparseArrays.SparseMatrixCSC{Float64, Int64} with 40190 stored entries:
⎡⣿⡟⠀⠀⢸⣷⣶⠀⠀⠲⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠛⠃⠀⠀⢸⣿⣶⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠛⠂⠀⠀⠘⠛⠛⠀⠀⠀⠀⠀⠈⠓⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [11]:
showElementResults(C.n, name="n")

1

In [12]:
showElementResults(C.t1, name="t1")

2

In [13]:
showElementResults(C.t2, name="t2")

3

In [14]:
showElementResults(C.gap, name="gap", visible=false)

4

In [15]:
C.active

1149-element BitVector:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 1
 0
 1
 1
 1
 1
 0
 1
 0
 0
 0
 1

In [16]:
C.state

1149-element Vector{UInt8}:
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
    ⋮
 0x01
 0x00
 0x01
 0x01
 0x01
 0x01
 0x00
 0x01
 0x00
 0x00
 0x00
 0x01

In [17]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
